In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# Web search tool implementation in an agent. 
from langchain.tools import tool
from langchain.agents import create_agent
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def search_web(query: str) -> Dict[str, Any]:
    """search the web for information."""
    return tavily_client.search(query)

agent = create_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[search_web],
)


In [ ]:
agent.invoke(
    {"messages": [{"role": "user", "content": "Who is the current CEO of DP World Logistics?"}]}
)

In [5]:
# Adding short-term memory agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

# define the Tavily client for web search
tavily_client = TavilyClient()

# create tool for web search
@tool
def search_web(query: str) -> Dict[str, Any]:
    """Search the web for accurate information."""
    return tavily_client.search(query)

agent = create_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[search_web],
    checkpointer=InMemorySaver()
)

# creates a config that groups the chat with thread ids
config = {"configurable": {"thread_id": "1"}}


In [7]:
# first run to fetch the latest information
agent.invoke(
    {"messages": [{"role": "user", "content": "Who is the current CEO of DP World Logistics?"}]},
    config
)

{'messages': [HumanMessage(content='Who is the current CEO of DP World Logistics?', additional_kwargs={}, response_metadata={}, id='f59b44bc-c5ea-42c1-af2a-9215fe625563'),
  AIMessage(content=[{'id': 'toolu_01HrX5X226ibezw231wTj63T', 'input': {'query': 'current CEO DP World Logistics'}, 'name': 'search_web', 'type': 'tool_use', 'caller': {'type': 'direct'}}], additional_kwargs={}, response_metadata={'id': 'msg_01VgvZ1TqWWS49oZszWSvNEr', 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 569, 'output_tokens': 59, 'server_tool_use': None, 'service_tier': 'standard', 'inference_geo': 'not_available'}, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_run--019c5cda-6cac-7b90-ba99-06d694e7d0ee-0', tool_calls=[{'name': 'search_web', 'args': {'query': 

In [8]:
# second run to see what more can it give me based on the first run
agent.invoke(
    {"messages": [{"role": "user", "content": "elaborate more on this."}]}, 
    config
)

{'messages': [HumanMessage(content='Who is the current CEO of DP World Logistics?', additional_kwargs={}, response_metadata={}, id='f59b44bc-c5ea-42c1-af2a-9215fe625563'),
  AIMessage(content=[{'id': 'toolu_01HrX5X226ibezw231wTj63T', 'input': {'query': 'current CEO DP World Logistics'}, 'name': 'search_web', 'type': 'tool_use', 'caller': {'type': 'direct'}}], additional_kwargs={}, response_metadata={'id': 'msg_01VgvZ1TqWWS49oZszWSvNEr', 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 569, 'output_tokens': 59, 'server_tool_use': None, 'service_tier': 'standard', 'inference_geo': 'not_available'}, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_run--019c5cda-6cac-7b90-ba99-06d694e7d0ee-0', tool_calls=[{'name': 'search_web', 'args': {'query': 

In [ ]:
# multimodal time RIP. 
# we now add capabilities to upload image to the agent. 

from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [12]:
print(uploader.value)

({'name': '58.png', 'type': 'image/png', 'size': 862658, 'content': <memory at 0x1131a8340>, 'last_modified': datetime.datetime(2024, 4, 11, 16, 26, 47, 745000, tzinfo=datetime.timezone.utc)},)


In [ ]:
# do the base64 encoding of the image. 
# P.S. the image should not exceed a max of 5MB or it throws an error.
import base64

uploaded_file = uploader.value[0]

content_mv = uploaded_file["content"]

img_bytes = bytes(content_mv)

img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [14]:
from langchain.agents import create_agent
agent = create_agent(
    model="claude-sonnet-4-5-20250929"
)

In [ ]:
# pass the multimodal question to the agent. 
agent.invoke(
    {"messages": [
        {"role": "user", 
         "content": [
             {"type": "text", "text": "tell me about this image."},
             {"type": "image", "base64": img_b64, "mime_type": "image/png"}
             ]
        }
    ]}
)

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'tell me about this image.'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/png'}], additional_kwargs={}, response_metadata={}, id='03ebea68-372d-4828-a75f-38d00016edce'),
  AIMessage(content="This is a striking digital artwork featuring a human skull rendered in a futuristic, cyberpunk aesthetic. Here are the key elements:\n\n**Visual Style:**\n- The skull appears translucent or X-ray-like, with a ghostly white/gray appearance\n- It's surrounded by bright pink/magenta neon geometric lines and circular patterns\n- The design has a glitch or digital distortion effect with horizontal scan lines\n- Motion blur or speed lines trail behind the skull, suggesting movement through space\n\n**Background:**\n- Deep black space filled with white stars and some pink-tinted stars\n- Creates a cosmic or outer space setting\n- The starfield gives depth and emphasizes the floating nature of the skull\n\n**Artistic Theme:**\n- Blends

In [19]:
# time to add capability to add sound to the agent
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm 

duration = 5 
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)

for _ in tqdm(range(duration * 10)):
    time.sleep(0.1)
sd.wait()
print("Done.")

buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.54it/s]


Done.


In [ ]:
# create agent to take the sound as input
agent = create_agent(
    # model="claude-sonnet-4-5-20250929" -- DID NOT WORK AS AUDIO MODEL. NOT SUPPORTED. 
    # model="claude-haiku-4-5-20251001" -- DID NOT WORK AS WELL....
    # model="claude-3-5-sonnet-20240620" -- DID NOT WORK RIP. 
)

agent.invoke(
    {"messages": [{
        "role": "user",
        "content": [
            {"type": "text", "text": "tell me about the audio file."},
            {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
        ]}
    ]}
)

# No models working for audio yet.
# skipping to not waste time on this yet. 

ValueError: Block of type audio is not supported.